In [1]:
"""
Timing Test — CPU vs GPU
=========================
Self-contained: only needs efficient-kan, torch, numpy, pandas.
No custom modules required.

Run from: Code/KAN_Section/Dense_vs_Sparse_KAN/
"""

# %% [markdown]
# # Timing Test: Dense KAN Epoch Speed
# Tests different configurations to estimate total experiment time.

# %%
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import time
from pathlib import Path
from torch.utils.data import TensorDataset, DataLoader
from efficient_kan import KAN

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

# %% [markdown]
# ## Load Data
# Tries real data first, falls back to synthetic if not found.

# %%
SPLITS_DIR = Path("../../..") / "Data" / "Splits"

def load_real_data(feature_set="full_moments", split="Split_A"):
    """Load real parquet data. Returns None if not found."""
    split_dir = SPLITS_DIR / split
    train_path = split_dir / f"{feature_set}_train.parquet"
    val_path = split_dir / f"{feature_set}_val.parquet"

    if not train_path.exists():
        print(f"  Not found: {train_path}")
        return None

    train_df = pd.read_parquet(train_path)
    val_df = pd.read_parquet(val_path)

    meta_cols = {"date", "target_daily_return", "minret_5d", "y_binary"}
    feat_cols = [c for c in train_df.columns if c not in meta_cols]

    X_train = torch.tensor(train_df[feat_cols].to_numpy(dtype=np.float32))
    y_train = torch.tensor(train_df["y_binary"].to_numpy(dtype=np.float32)).unsqueeze(1)
    X_val = torch.tensor(val_df[feat_cols].to_numpy(dtype=np.float32))
    y_val = torch.tensor(val_df["y_binary"].to_numpy(dtype=np.float32)).unsqueeze(1)

    print(f"  Loaded {feature_set}: train {X_train.shape}, val {X_val.shape}")
    return X_train, y_train, X_val, y_val

def load_or_generate(feature_set="full_moments"):
    """Load real data or generate synthetic matching dimensions."""
    real = load_real_data(feature_set)
    if real is not None:
        return real

    dims = {"full_moments": 2212, "means_only": 722}
    n_feat = dims[feature_set]
    print(f"  Generating synthetic: {feature_set} ({n_feat} features)")
    X_train = torch.randn(3000, n_feat)
    y_train = torch.randint(0, 2, (3000, 1)).float()
    X_val = torch.randn(500, n_feat)
    y_val = torch.randint(0, 2, (500, 1)).float()
    return X_train, y_train, X_val, y_val

# Load both feature sets
print("Loading data...")
data_full = load_or_generate("full_moments")
data_means = load_or_generate("means_only")

# %% [markdown]
# ## Timing Function

# %%
def time_epochs(model, train_loader, val_loader, device, n_warmup=2, n_timed=10):
    """
    Time n_timed training epochs (after warmup).
    Each epoch = full forward + backward + optimizer step + validation pass.
    """
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

    model.to(device)

    # Warmup (not timed — lets GPU caches and JIT settle)
    model.train()
    for _ in range(n_warmup):
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(Xb), yb)
            loss.backward()
            optimizer.step()

    if device.type == "cuda":
        torch.cuda.synchronize()

    # Timed epochs
    epoch_times = []
    for i in range(n_timed):
        start = time.perf_counter()

        # Training pass
        model.train()
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(Xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        # Validation pass (train_model does this every epoch for early stopping)
        model.eval()
        with torch.no_grad():
            for Xb, yb in val_loader:
                Xb, yb = Xb.to(device), yb.to(device)
                _ = model(Xb)

        if device.type == "cuda":
            torch.cuda.synchronize()

        elapsed = time.perf_counter() - start
        epoch_times.append(elapsed)

    return epoch_times

# %% [markdown]
# ## Run Timing Tests

# %%
# Configurations to test
configs = [
    # (name, feature_set, grid_size, batch_size)
    ("full_moments G5  bs128", "full_moments", 5, 128),
    ("full_moments G8  bs128", "full_moments", 8, 128),
    ("full_moments G12 bs128", "full_moments", 12, 128),
    ("means_only   G5  bs128", "means_only", 5, 128),
    ("means_only   G8  bs128", "means_only", 8, 128),
    ("means_only   G12 bs128", "means_only", 12, 128),
    # Batch size variants (full_moments G8 only)
    ("full_moments G8  bs64 ", "full_moments", 8, 64),
    ("full_moments G8  bs256", "full_moments", 8, 256),
]

# Which devices to test
devices = [torch.device("cpu")]
if torch.cuda.is_available():
    devices.append(torch.device("cuda"))

# Data lookup
data_lookup = {
    "full_moments": data_full,
    "means_only": data_means,
}

# Store all results
results = []

for device in devices:
    print(f"\n{'='*70}")
    print(f"  DEVICE: {device}")
    print(f"{'='*70}")

    for name, feat_set, grid_size, batch_size in configs:
        X_train, y_train, X_val, y_val = data_lookup[feat_set]
        n_features = X_train.shape[1]

        train_loader = DataLoader(
            TensorDataset(X_train, y_train),
            batch_size=batch_size, shuffle=True,
        )
        val_loader = DataLoader(
            TensorDataset(X_val, y_val),
            batch_size=batch_size,
        )

        # Create model
        model = KAN(
            [n_features, 101, 14, 1],
            grid_size=grid_size,
            spline_order=3,
        )
        n_params = sum(p.numel() for p in model.parameters())

        print(f"\n  {name}  |  {n_params:>10,} params  |  {len(train_loader)} batches/epoch")

        try:
            times = time_epochs(model, train_loader, val_loader, device,
                                n_warmup=2, n_timed=10)

            mean_t = np.mean(times)
            std_t = np.std(times)

            # Estimates (assuming ~100 epochs per trial before early stopping)
            trial_min = mean_t * 100 / 60
            optuna_60_hrs = mean_t * 100 * 60 / 3600
            warm_15_hrs = mean_t * 100 * 15 / 3600

            print(f"    Epoch: {mean_t:.3f}s ± {std_t:.3f}s")
            print(f"    Est. 1 trial (~100 epochs):    {trial_min:.1f} min")
            print(f"    Est. 60 trials (full Optuna):   {optuna_60_hrs:.1f} hrs")
            print(f"    Est. 15 trials (warm-start):    {warm_15_hrs:.1f} hrs")

            results.append({
                "device": str(device),
                "config": name.strip(),
                "n_params": n_params,
                "mean_epoch_s": mean_t,
                "std_epoch_s": std_t,
                "est_trial_min": trial_min,
                "est_60trials_hrs": optuna_60_hrs,
                "est_15trials_hrs": warm_15_hrs,
            })

        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                print(f"    ✗ OUT OF MEMORY")
                if device.type == "cuda":
                    torch.cuda.empty_cache()
            else:
                raise

        # Clean up
        del model
        if device.type == "cuda":
            torch.cuda.empty_cache()

# %% [markdown]
# ## Summary Table

# %%
print(f"\n{'='*70}")
print(f"  SUMMARY TABLE")
print(f"{'='*70}\n")

print(f"  {'Config':<28} {'Device':<7} {'Params':>10} "
      f"{'Epoch(s)':>9} {'1 Trial':>9} {'60 Trial':>9} {'15 Trial':>9}")
print(f"  {'':28} {'':7} {'':10} {'':>9} {'(min)':>9} {'(hrs)':>9} {'(hrs)':>9}")
print("  " + "-" * 90)

for r in results:
    print(f"  {r['config']:<28} {r['device']:<7} {r['n_params']:>10,} "
          f"{r['mean_epoch_s']:>9.3f} {r['est_trial_min']:>9.1f} "
          f"{r['est_60trials_hrs']:>9.1f} {r['est_15trials_hrs']:>9.1f}")

# GPU speedup
cpu_results = {r["config"]: r for r in results if r["device"] == "cpu"}
gpu_results = {r["config"]: r for r in results if r["device"] == "cuda"}
if gpu_results:
    print(f"\n  GPU Speedup:")
    for cfg in cpu_results:
        if cfg in gpu_results:
            speedup = cpu_results[cfg]["mean_epoch_s"] / gpu_results[cfg]["mean_epoch_s"]
            print(f"    {cfg}: {speedup:.1f}x")

# %% [markdown]
# ## Experiment Planning
# 5 models × 4 splits. How long will it all take?

# %%
# Use the config you're most likely to run (adjust if needed)
REF_CONFIG = "full_moments G8  bs128"
best_device = "cuda" if gpu_results else "cpu"

ref = [r for r in results if r["config"] == REF_CONFIG and r["device"] == best_device]
if not ref:
    ref = [r for r in results if "full_moments" in r["config"] and r["device"] == best_device]

if ref:
    epoch_s = ref[0]["mean_epoch_s"]
    n_models = 5  # Ridge, DenseKAN, SparseKAN, SparseKAN-MF, DenseMLP, SparseMLP
    # (Ridge is sklearn, near-instant — so effectively 5 neural models)

    print(f"\n{'='*70}")
    print(f"  EXPERIMENT PLANNING (ref: {ref[0]['config']} on {best_device})")
    print(f"  Epoch time: {epoch_s:.3f}s")
    print(f"{'='*70}\n")

    scenarios = {
        "A: Full Optuna everywhere": {
            "full_splits": 4, "full_trials": 60,
            "warm_splits": 0, "warm_trials": 0,
        },
        "B: Optuna on A, warm-start B/C/D": {
            "full_splits": 1, "full_trials": 60,
            "warm_splits": 3, "warm_trials": 15,
        },
        "C: Optuna on A+C, warm-start B+D": {
            "full_splits": 2, "full_trials": 60,
            "warm_splits": 2, "warm_trials": 15,
        },
        "D: Optuna on A, fixed B/C/D": {
            "full_splits": 1, "full_trials": 60,
            "warm_splits": 0, "warm_trials": 0,
            "fixed_splits": 3, "fixed_trials": 1,
        },
    }

    epochs_per_trial = 100  # approximate, depends on early stopping

    for name, s in scenarios.items():
        full_hrs = n_models * s["full_splits"] * s["full_trials"] * epochs_per_trial * epoch_s / 3600
        warm_hrs = n_models * s["warm_splits"] * s["warm_trials"] * epochs_per_trial * epoch_s / 3600
        fixed_hrs = n_models * s.get("fixed_splits", 0) * s.get("fixed_trials", 0) * epochs_per_trial * epoch_s / 3600
        total = full_hrs + warm_hrs + fixed_hrs

        print(f"  {name}")
        print(f"    Full Optuna: {full_hrs:>6.1f} hrs")
        if warm_hrs > 0:
            print(f"    Warm-start:  {warm_hrs:>6.1f} hrs")
        if fixed_hrs > 0:
            print(f"    Fixed:       {fixed_hrs:>6.1f} hrs")
        print(f"    TOTAL:       {total:>6.1f} hrs  ({total/24:.1f} days)\n")

    print(f"  Available: ~3 weeks ≈ 21 days")
    print(f"  Note: Ridge is sklearn (seconds). Dense MLP is much faster than KAN.")
    print(f"  Real time will be lower since not all models are Dense KAN sized.")
    print(f"  Sparse KAN has ~100x fewer params → proportionally faster epochs.")
else:
    print("  Could not find reference config for planning.")

# %%

ModuleNotFoundError: No module named 'efficient_kan'

# GPU

In [5]:
"""
Timing Test — CPU vs GPU (Colab Version)
==========================================
Run from VSCode connected to a Colab runtime.
Installs dependencies in the runtime automatically.
"""

# %% [markdown]
# # Timing Test: Dense KAN — CPU vs GPU
# Install dependencies, load data, time epochs, plan experiments.

# %%
# ── Install in Colab runtime ──
!pip install -q git+https://github.com/Blealtan/efficient-kan.git
# %%
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import time
from pathlib import Path
from torch.utils.data import TensorDataset, DataLoader
from efficient_kan import KAN

print(f"PyTorch: {torch.__version__}")
print(f"NumPy:   {np.__version__}")
print(f"CUDA:    {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:     {torch.cuda.get_device_name(0)}")
    print(f"VRAM:    {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# %% [markdown]
# ## Load Data
# Option 1: Mount Google Drive (if you uploaded Data/Splits/ there)
# Option 2: Use synthetic data matching your real dimensions

# %%
# ── Set USE_DRIVE = True if your data is on Google Drive ──
USE_DRIVE = False

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    # Adjust this path to where you put Data/Splits/ on Drive
    SPLITS_DIR = Path("/content/drive/MyDrive/Thesis/Data/Splits")
else:
    SPLITS_DIR = None  # will use synthetic data

def load_real_data(feature_set="full_moments", split="Split_A"):
    """Load real parquet data. Returns None if not found."""
    if SPLITS_DIR is None:
        return None

    split_dir = SPLITS_DIR / split
    train_path = split_dir / f"{feature_set}_train.parquet"
    val_path = split_dir / f"{feature_set}_val.parquet"

    if not train_path.exists():
        print(f"  Not found: {train_path}")
        return None

    train_df = pd.read_parquet(train_path)
    val_df = pd.read_parquet(val_path)

    meta_cols = {"date", "target_daily_return", "minret_5d", "y_binary"}
    feat_cols = [c for c in train_df.columns if c not in meta_cols]

    X_train = torch.tensor(train_df[feat_cols].to_numpy(dtype=np.float32))
    y_train = torch.tensor(train_df["y_binary"].to_numpy(dtype=np.float32)).unsqueeze(1)
    X_val = torch.tensor(val_df[feat_cols].to_numpy(dtype=np.float32))
    y_val = torch.tensor(val_df["y_binary"].to_numpy(dtype=np.float32)).unsqueeze(1)

    print(f"  Loaded {feature_set}: train {X_train.shape}, val {X_val.shape}")
    return X_train, y_train, X_val, y_val

def generate_synthetic(feature_set="full_moments"):
    """Generate synthetic data matching real dimensions."""
    dims = {"full_moments": 2212, "means_only": 722}
    n_feat = dims[feature_set]
    # Match real data: ~3000 train, ~500 val, ~18% crash rate
    n_train, n_val = 3000, 500
    crash_rate = 0.18

    X_train = torch.randn(n_train, n_feat)
    y_train = (torch.rand(n_train) < crash_rate).float().unsqueeze(1)
    X_val = torch.randn(n_val, n_feat)
    y_val = (torch.rand(n_val) < crash_rate).float().unsqueeze(1)

    print(f"  Synthetic {feature_set}: train ({n_train}, {n_feat}), val ({n_val}, {n_feat})")
    return X_train, y_train, X_val, y_val

def load_or_generate(feature_set):
    real = load_real_data(feature_set)
    return real if real is not None else generate_synthetic(feature_set)

print("Loading data...")
data_full = load_or_generate("full_moments")
data_means = load_or_generate("means_only")

# %% [markdown]
# ## Timing Function

# %%
def time_epochs(model, train_loader, val_loader, device, n_warmup=2, n_timed=10):
    """
    Time n_timed training epochs (after warmup).
    Each timed epoch = train pass + val pass (matches real training loop).
    """
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    model.to(device)

    # Warmup
    model.train()
    for _ in range(n_warmup):
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(Xb), yb)
            loss.backward()
            optimizer.step()

    if device.type == "cuda":
        torch.cuda.synchronize()

    # Timed epochs
    epoch_times = []
    for i in range(n_timed):
        start = time.perf_counter()

        model.train()
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(Xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        model.eval()
        with torch.no_grad():
            for Xb, yb in val_loader:
                Xb, yb = Xb.to(device), yb.to(device)
                _ = model(Xb)

        if device.type == "cuda":
            torch.cuda.synchronize()

        epoch_times.append(time.perf_counter() - start)

    return epoch_times

# %% [markdown]
# ## Run All Configurations

# %%
configs = [
    ("full_moments G5  bs128", "full_moments", 5, 128),
    ("full_moments G8  bs128", "full_moments", 8, 128),
    ("full_moments G12 bs128", "full_moments", 12, 128),
    ("means_only   G5  bs128", "means_only", 5, 128),
    ("means_only   G8  bs128", "means_only", 8, 128),
    ("means_only   G12 bs128", "means_only", 12, 128),
    ("full_moments G8  bs64 ", "full_moments", 8, 64),
    ("full_moments G8  bs256", "full_moments", 8, 256),
]

devices = [torch.device("cpu")]
if torch.cuda.is_available():
    devices.append(torch.device("cuda"))

data_lookup = {"full_moments": data_full, "means_only": data_means}
results = []

for device in devices:
    print(f"\n{'='*70}")
    print(f"  DEVICE: {device}" + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""))
    print(f"{'='*70}")

    for name, feat_set, grid_size, batch_size in configs:
        X_train, y_train, X_val, y_val = data_lookup[feat_set]
        n_features = X_train.shape[1]

        train_loader = DataLoader(
            TensorDataset(X_train, y_train),
            batch_size=batch_size, shuffle=True,
        )
        val_loader = DataLoader(
            TensorDataset(X_val, y_val),
            batch_size=batch_size,
        )

        model = KAN([n_features, 101, 14, 1], grid_size=grid_size, spline_order=3)
        n_params = sum(p.numel() for p in model.parameters())

        print(f"\n  {name}  |  {n_params:>10,} params  |  {len(train_loader)} batches")

        try:
            times = time_epochs(model, train_loader, val_loader, device,
                                n_warmup=2, n_timed=10)

            mean_t = np.mean(times)
            std_t = np.std(times)
            trial_min = mean_t * 100 / 60
            optuna_25_hrs = mean_t * 100 * 25 / 3600

            print(f"    Epoch: {mean_t:.3f}s ± {std_t:.3f}s")
            print(f"    Est. 1 trial (~100 epochs): {trial_min:.1f} min")
            print(f"    Est. 25 trials (Optuna):    {optuna_25_hrs:.1f} hrs")

            results.append({
                "device": str(device),
                "config": name.strip(),
                "n_params": n_params,
                "mean_s": mean_t,
                "std_s": std_t,
                "trial_min": trial_min,
                "optuna_25_hrs": optuna_25_hrs,
            })

        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                print(f"    ✗ OUT OF MEMORY")
                if device.type == "cuda":
                    torch.cuda.empty_cache()
            else:
                raise

        del model
        if device.type == "cuda":
            torch.cuda.empty_cache()

# %% [markdown]
# ## Summary

# %%
print(f"\n{'='*70}")
print(f"  SUMMARY")
print(f"{'='*70}\n")

print(f"  {'Config':<28} {'Device':<7} {'Params':>10} {'Epoch':>8} {'1 Trial':>9} {'25 Opt':>9}")
print(f"  {'':28} {'':7} {'':10} {'(sec)':>8} {'(min)':>9} {'(hrs)':>9}")
print("  " + "-" * 80)

for r in results:
    print(f"  {r['config']:<28} {r['device']:<7} {r['n_params']:>10,} "
          f"{r['mean_s']:>8.3f} {r['trial_min']:>9.1f} {r['optuna_25_hrs']:>9.1f}")

# GPU speedup
cpu_res = {r["config"]: r for r in results if r["device"] == "cpu"}
gpu_res = {r["config"]: r for r in results if r["device"] == "cuda"}
if gpu_res:
    print(f"\n  GPU Speedup:")
    for cfg in cpu_res:
        if cfg in gpu_res:
            speedup = cpu_res[cfg]["mean_s"] / gpu_res[cfg]["mean_s"]
            print(f"    {cfg}: {speedup:.1f}x")

# %% [markdown]
# ## Experiment Planning

# %%
# Find a reliable reference (full_moments, any G, prefer GPU)
best_device = "cuda" if gpu_res else "cpu"
ref_candidates = [r for r in results
                  if "full_moments" in r["config"]
                  and "G8" in r["config"]
                  and "bs128" in r["config"]
                  and r["device"] == best_device]
if not ref_candidates:
    ref_candidates = [r for r in results if "full_moments" in r["config"] and r["device"] == best_device]

if ref_candidates:
    ref = ref_candidates[0]
    epoch_s = ref["mean_s"]
    n_models = 5  # DenseKAN, SparseKAN, SparseKAN-MF, DenseMLP, SparseMLP (Ridge is instant)
    epochs_per_trial = 100

    print(f"\n{'='*70}")
    print(f"  PLANNING: 5 neural models × 4 splits")
    print(f"  Reference: {ref['config']} on {best_device} = {epoch_s:.3f}s/epoch")
    print(f"{'='*70}\n")

    # Scenario: Optuna on 2 splits (A+C), fixed on 2 (B+D)
    # 25 trials per Optuna run
    optuna_hrs = n_models * 2 * 25 * epochs_per_trial * epoch_s / 3600
    fixed_hrs = n_models * 2 * 1 * epochs_per_trial * epoch_s / 3600
    total = optuna_hrs + fixed_hrs

    print(f"  Your plan: Optuna (25 trials) on A+C, fixed on B+D")
    print(f"    Optuna (2 splits): {optuna_hrs:>6.1f} hrs")
    print(f"    Fixed (2 splits):  {fixed_hrs:>6.1f} hrs")
    print(f"    TOTAL:             {total:>6.1f} hrs  ({total/24:.1f} days)")
    print()

    # Context: the Dense KAN is the SLOWEST model
    # Sparse KAN ~100x fewer params, MLP has no spline overhead
    print(f"  Note: {epoch_s:.2f}s/epoch is the WORST case (Dense KAN full_moments).")
    print(f"  Sparse KAN (~12K params vs ~2.9M) will be much faster.")
    print(f"  Dense MLP (no spline computation) will also be faster.")
    print(f"  Realistic total is probably 40-60% of the estimate above.")
    print(f"\n  Available: ~3 weeks")
else:
    print("  No reference config found.")

# %%

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
PyTorch: 2.11.0+cu128
NumPy:   2.0.2
CUDA:    True
GPU:     Tesla T4
VRAM:    15.6 GB
Loading data...
  Synthetic full_moments: train (3000, 2212), val (500, 2212)
  Synthetic means_only: train (3000, 722), val (500, 722)

  DEVICE: cpu

  full_moments G5  bs128  |   2,248,400 params  |  24 batches
    Epoch: 3.606s ± 0.322s
    Est. 1 trial (~100 epochs): 6.0 min
    Est. 25 trials (Optuna):    2.5 hrs

  full_moments G8  bs128  |   2,922,920 params  |  24 batches
    Epoch: 4.260s ± 0.383s
    Est. 1 trial (~100 epochs): 7.1 min
    Est. 25 trials (Optuna):    3.0 hrs

  full_moments G12 bs128  |   3,822,280 params  |  24 batches
    Epoch: 5.499s ± 0.255s
    Est. 1 trial (~100 epochs): 9.2 min
    Est. 25 trials (Optuna):    3.8 hrs

  means_only   G5  bs128  |     743,500 params  |  24 batches
    Epoch: 1.144s ± 0.197s
    Est. 1 trial (~100 epoch

In [1]:
"""
Timing Test — CPU vs GPU (Colab Version)
==========================================
Run from VSCode connected to a Colab runtime.
Installs dependencies in the runtime automatically.
"""

# %% [markdown]
# # Timing Test: Dense KAN — CPU vs GPU
# Install dependencies, load data, time epochs, plan experiments.

# %%
# ── Install in Colab runtime ──
!pip install -q git+https://github.com/Blealtan/efficient-kan.git
# %%
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import time
from pathlib import Path
from torch.utils.data import TensorDataset, DataLoader
from efficient_kan import KAN

print(f"PyTorch: {torch.__version__}")
print(f"NumPy:   {np.__version__}")
print(f"CUDA:    {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:     {torch.cuda.get_device_name(0)}")
    print(f"VRAM:    {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# %% [markdown]
# ## Load Data
# Option 1: Mount Google Drive (if you uploaded Data/Splits/ there)
# Option 2: Use synthetic data matching your real dimensions

# %%
# ── Set USE_DRIVE = True if your data is on Google Drive ──
USE_DRIVE = False

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    # Adjust this path to where you put Data/Splits/ on Drive
    SPLITS_DIR = Path("/content/drive/MyDrive/Thesis/Data/Splits")
else:
    SPLITS_DIR = None  # will use synthetic data

def load_real_data(feature_set="full_moments", split="Split_A"):
    """Load real parquet data. Returns None if not found."""
    if SPLITS_DIR is None:
        return None

    split_dir = SPLITS_DIR / split
    train_path = split_dir / f"{feature_set}_train.parquet"
    val_path = split_dir / f"{feature_set}_val.parquet"

    if not train_path.exists():
        print(f"  Not found: {train_path}")
        return None

    train_df = pd.read_parquet(train_path)
    val_df = pd.read_parquet(val_path)

    meta_cols = {"date", "target_daily_return", "minret_5d", "y_binary"}
    feat_cols = [c for c in train_df.columns if c not in meta_cols]

    X_train = torch.tensor(train_df[feat_cols].to_numpy(dtype=np.float32))
    y_train = torch.tensor(train_df["y_binary"].to_numpy(dtype=np.float32)).unsqueeze(1)
    X_val = torch.tensor(val_df[feat_cols].to_numpy(dtype=np.float32))
    y_val = torch.tensor(val_df["y_binary"].to_numpy(dtype=np.float32)).unsqueeze(1)

    print(f"  Loaded {feature_set}: train {X_train.shape}, val {X_val.shape}")
    return X_train, y_train, X_val, y_val

def generate_synthetic(feature_set="full_moments"):
    """Generate synthetic data matching real dimensions."""
    dims = {"full_moments": 2212, "means_only": 722}
    n_feat = dims[feature_set]
    # Match real data: ~3000 train, ~500 val, ~18% crash rate
    n_train, n_val = 3000, 500
    crash_rate = 0.18

    X_train = torch.randn(n_train, n_feat)
    y_train = (torch.rand(n_train) < crash_rate).float().unsqueeze(1)
    X_val = torch.randn(n_val, n_feat)
    y_val = (torch.rand(n_val) < crash_rate).float().unsqueeze(1)

    print(f"  Synthetic {feature_set}: train ({n_train}, {n_feat}), val ({n_val}, {n_feat})")
    return X_train, y_train, X_val, y_val

def load_or_generate(feature_set):
    real = load_real_data(feature_set)
    return real if real is not None else generate_synthetic(feature_set)

print("Loading data...")
data_full = load_or_generate("full_moments")
data_means = load_or_generate("means_only")

# %% [markdown]
# ## Timing Function

# %%
def time_epochs(model, train_loader, val_loader, device, n_warmup=2, n_timed=10):
    """
    Time n_timed training epochs (after warmup).
    Each timed epoch = train pass + val pass (matches real training loop).
    """
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    model.to(device)

    # Warmup
    model.train()
    for _ in range(n_warmup):
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(Xb), yb)
            loss.backward()
            optimizer.step()

    if device.type == "cuda":
        torch.cuda.synchronize()

    # Timed epochs
    epoch_times = []
    for i in range(n_timed):
        start = time.perf_counter()

        model.train()
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(Xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        model.eval()
        with torch.no_grad():
            for Xb, yb in val_loader:
                Xb, yb = Xb.to(device), yb.to(device)
                _ = model(Xb)

        if device.type == "cuda":
            torch.cuda.synchronize()

        epoch_times.append(time.perf_counter() - start)

    return epoch_times

# %% [markdown]
# ## Run All Configurations

# %%
configs = [
    ("full_moments G5  bs128", "full_moments", 5, 128),
    ("full_moments G8  bs128", "full_moments", 8, 128),
    ("full_moments G12 bs128", "full_moments", 12, 128),
    ("means_only   G5  bs128", "means_only", 5, 128),
    ("means_only   G8  bs128", "means_only", 8, 128),
    ("means_only   G12 bs128", "means_only", 12, 128),
    ("full_moments G8  bs64 ", "full_moments", 8, 64),
    ("full_moments G8  bs256", "full_moments", 8, 256),
]

devices = [torch.device("cpu")]
if torch.cuda.is_available():
    devices.append(torch.device("cuda"))

data_lookup = {"full_moments": data_full, "means_only": data_means}
results = []

for device in devices:
    print(f"\n{'='*70}")
    print(f"  DEVICE: {device}" + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""))
    print(f"{'='*70}")

    for name, feat_set, grid_size, batch_size in configs:
        X_train, y_train, X_val, y_val = data_lookup[feat_set]
        n_features = X_train.shape[1]

        train_loader = DataLoader(
            TensorDataset(X_train, y_train),
            batch_size=batch_size, shuffle=True,
        )
        val_loader = DataLoader(
            TensorDataset(X_val, y_val),
            batch_size=batch_size,
        )

        model = KAN([n_features, 101, 14, 1], grid_size=grid_size, spline_order=3)
        n_params = sum(p.numel() for p in model.parameters())

        print(f"\n  {name}  |  {n_params:>10,} params  |  {len(train_loader)} batches")

        try:
            times = time_epochs(model, train_loader, val_loader, device,
                                n_warmup=2, n_timed=10)

            mean_t = np.mean(times)
            std_t = np.std(times)
            trial_min = mean_t * 100 / 60
            optuna_25_hrs = mean_t * 100 * 25 / 3600

            print(f"    Epoch: {mean_t:.3f}s ± {std_t:.3f}s")
            print(f"    Est. 1 trial (~100 epochs): {trial_min:.1f} min")
            print(f"    Est. 25 trials (Optuna):    {optuna_25_hrs:.1f} hrs")

            results.append({
                "device": str(device),
                "config": name.strip(),
                "n_params": n_params,
                "mean_s": mean_t,
                "std_s": std_t,
                "trial_min": trial_min,
                "optuna_25_hrs": optuna_25_hrs,
            })

        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                print(f"    ✗ OUT OF MEMORY")
                if device.type == "cuda":
                    torch.cuda.empty_cache()
            else:
                raise

        del model
        if device.type == "cuda":
            torch.cuda.empty_cache()

# %% [markdown]
# ## Summary

# %%
print(f"\n{'='*70}")
print(f"  SUMMARY")
print(f"{'='*70}\n")

print(f"  {'Config':<28} {'Device':<7} {'Params':>10} {'Epoch':>8} {'1 Trial':>9} {'25 Opt':>9}")
print(f"  {'':28} {'':7} {'':10} {'(sec)':>8} {'(min)':>9} {'(hrs)':>9}")
print("  " + "-" * 80)

for r in results:
    print(f"  {r['config']:<28} {r['device']:<7} {r['n_params']:>10,} "
          f"{r['mean_s']:>8.3f} {r['trial_min']:>9.1f} {r['optuna_25_hrs']:>9.1f}")

# GPU speedup
cpu_res = {r["config"]: r for r in results if r["device"] == "cpu"}
gpu_res = {r["config"]: r for r in results if r["device"] == "cuda"}
if gpu_res:
    print(f"\n  GPU Speedup:")
    for cfg in cpu_res:
        if cfg in gpu_res:
            speedup = cpu_res[cfg]["mean_s"] / gpu_res[cfg]["mean_s"]
            print(f"    {cfg}: {speedup:.1f}x")

# %% [markdown]
# ## Experiment Planning

# %%
# Find a reliable reference (full_moments, any G, prefer GPU)
best_device = "cuda" if gpu_res else "cpu"
ref_candidates = [r for r in results
                  if "full_moments" in r["config"]
                  and "G8" in r["config"]
                  and "bs128" in r["config"]
                  and r["device"] == best_device]
if not ref_candidates:
    ref_candidates = [r for r in results if "full_moments" in r["config"] and r["device"] == best_device]

if ref_candidates:
    ref = ref_candidates[0]
    epoch_s = ref["mean_s"]
    n_models = 5  # DenseKAN, SparseKAN, SparseKAN-MF, DenseMLP, SparseMLP (Ridge is instant)
    epochs_per_trial = 100

    print(f"\n{'='*70}")
    print(f"  PLANNING: 5 neural models × 4 splits")
    print(f"  Reference: {ref['config']} on {best_device} = {epoch_s:.3f}s/epoch")
    print(f"{'='*70}\n")

    # Scenario: Optuna on 2 splits (A+C), fixed on 2 (B+D)
    # 25 trials per Optuna run
    optuna_hrs = n_models * 2 * 25 * epochs_per_trial * epoch_s / 3600
    fixed_hrs = n_models * 2 * 1 * epochs_per_trial * epoch_s / 3600
    total = optuna_hrs + fixed_hrs

    print(f"  Your plan: Optuna (25 trials) on A+C, fixed on B+D")
    print(f"    Optuna (2 splits): {optuna_hrs:>6.1f} hrs")
    print(f"    Fixed (2 splits):  {fixed_hrs:>6.1f} hrs")
    print(f"    TOTAL:             {total:>6.1f} hrs  ({total/24:.1f} days)")
    print()

    # Context: the Dense KAN is the SLOWEST model
    # Sparse KAN ~100x fewer params, MLP has no spline overhead
    print(f"  Note: {epoch_s:.2f}s/epoch is the WORST case (Dense KAN full_moments).")
    print(f"  Sparse KAN (~12K params vs ~2.9M) will be much faster.")
    print(f"  Dense MLP (no spline computation) will also be faster.")
    print(f"  Realistic total is probably 40-60% of the estimate above.")
    print(f"\n  Available: ~3 weeks")
else:
    print("  No reference config found.")

# %%

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
PyTorch: 2.11.0+cpu
NumPy:   2.0.2
CUDA:    False
Loading data...
  Synthetic full_moments: train (3000, 2212), val (500, 2212)
  Synthetic means_only: train (3000, 722), val (500, 722)

  DEVICE: cpu

  full_moments G5  bs128  |   2,248,400 params  |  24 batches
    Epoch: 4.724s ± 0.431s
    Est. 1 trial (~100 epochs): 7.9 min
    Est. 25 trials (Optuna):    3.3 hrs

  full_moments G8  bs128  |   2,922,920 params  |  24 batches
    Epoch: 5.686s ± 0.428s
    Est. 1 trial (~100 epochs): 9.5 min
    Est. 25 trials (Optuna):    3.9 hrs

  full_moments G12 bs128  |   3,822,280 params  |  24 batches
    Epoch: 6.483s ± 0.381s
    Est. 1 trial (~100 epochs): 10.8 min
    Est. 25 trials (Optuna):    4.5 hrs

  means_only   G5  bs128  |     743,500 params  |  24 batches
    Epoch: 1.555s ± 0.244s
    Est. 1 trial (~100 epochs): 2.6 min
    Est. 25 trials (Opt